# 01 · Exploración de datos de Threat Intelligence (VeriThreat)

**Objetivo:** conocer la estructura, calidad y solapamiento de las fuentes que alimentan VeriThreat y fijar el esquema común de la base de datos. Este notebook refleja el **conjunto final de fuentes** tras evaluar y descartar las que aportaban poco.

| Fuente | Archivo | Papel |
|---|---|---|
| ThreatFox (abuse.ch) | `threatfox.csv` | **Núcleo.** IOCs mixtos con familia, confianza y etiquetas; única fuente con C2 |
| URLhaus (abuse.ch) | `urlhaus.csv` | **Núcleo.** URLs de distribución de malware con estado online/offline |
| blocklist.de | `export-ips_all.txt` | **Núcleo.** IPs que atacan servicios (única fuente grande de IPs) y /24 completos listados en bloque |
| PhishTank | `verified_online.csv` | **Núcleo.** Phishing verificado por la comunidad (histórico filtrado a 90 días) |
| TweetFeed | `month.csv` + `exclusive.csv` | **Núcleo.** IOCs de investigadores con reportero, etiquetas y enlace al tweet |
| IPsum | `ipsum_level1..8.txt` | **Señal de fiabilidad** para IPs: nº de listas en las que aparece (no aporta indicadores propios) |
| Tranco | `top-1m.csv` | Base de la clase *no maliciosa* |

**Descartadas** (ver el anexo y el documento de fase): MalwareBazaar, ThreatCluster, OTX, Feodo Tracker, Spamhaus, `IoCs_Identifiers.json`, filtro de Bloom de CIRCL, MISP como vía principal, vciso.au, C2IntelFeeds/C2-Tracker, FireHOL y el resto de listas de IPs de ataque.

> Las exportaciones tienen **ventanas temporales distintas** (sección 3). La ausencia de un indicador en una fuente no implica que sea inactivo ni benigno.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# cargadores_corregidos.py está en ai/src (o junto al notebook)
for _p in (Path.cwd(), Path.cwd().parent / "src"):
    if (_p / "cargadores_corregidos.py").exists():
        sys.path.insert(0, str(_p))
from cargadores_corregidos import (DOBLE_USO, descargar_ipsum, es_doble_uso, es_ip, load_blocklist, load_ipsum,
                                   load_phishtank, load_threatfox, load_tranco, load_tweetfeed, load_urlhaus, marcar_bloques_completos, registrable)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

# Carpeta con los CSV: la variable de entorno VERITHREAT_DATA tiene prioridad; después, la primera que exista.
CANDIDATAS = [Path(os.environ["VERITHREAT_DATA"])] if "VERITHREAT_DATA" in os.environ else []
CANDIDATAS += [Path("data/raw"), Path("../data/raw"), Path("data"), Path("."), Path("/mnt/user-data/uploads")]
DATA_DIR = next(p for p in CANDIDATAS if (p / "threatfox.csv").exists())
print("DATA_DIR =", DATA_DIR.resolve())

VENTANA_PHISHTANK_DIAS = 90   # PhishTank trae histórico desde 2011: nos quedamos con lo reciente
DESCARGAR_IPSUM = True        # descarga los ficheros de niveles de IPsum si no están (GitHub, sin clave)

## 1. Carga de datos

Cada fuente tiene un formato propio (los cargadores están en `cargadores_corregidos.py`):

- **URLhaus**: la cabecera es `# id,...` y los nulos vienen como `None`.
- **ThreatFox**: fechas con sufijo `UTC` y `tags` como texto de lista Python.
- **blocklist.de**: una IP por línea, sin cabecera, fecha ni categoría (IPv4 e IPv6).
- **PhishTank**: URLs con entidades HTML (`&amp;`) e histórico desde 2011.
- **TweetFeed**: sin cabecera; los dominios suelen ser solo el host de una URL del mismo tweet; `exclusive.csv` permite saber qué filas corroboran sus 5 feeds de referencia.
- **IPsum**: el fichero del nivel N contiene las IPs que aparecen en N o más listas.

In [ ]:
if DESCARGAR_IPSUM:
    try:
        print("IPsum descargado:", descargar_ipsum(DATA_DIR) or "ya estaba")
    except Exception as e:  # sin red o carpeta de solo lectura
        print("No se pudo descargar IPsum:", e)

tf = load_threatfox(DATA_DIR / "threatfox.csv")
uh = load_urlhaus(DATA_DIR / "urlhaus.csv")
bl = load_blocklist(DATA_DIR / "export-ips_all.txt")
bl["bloque"] = marcar_bloques_completos(bl["ip"])  # /24 completos listados en bloque (no son ataques individuales)
pt = load_phishtank(DATA_DIR / "verified_online.csv", ventana_dias=VENTANA_PHISHTANK_DIAS)
tw = load_tweetfeed(DATA_DIR / "month.csv", DATA_DIR / "exclusive.csv")
tr = load_tranco(DATA_DIR / "top-1m.csv")
ips = load_ipsum(DATA_DIR)

datasets = {"threatfox": tf, "urlhaus": uh, "blocklist": bl, "phishtank": pt, "tweetfeed": tw}
resumen = pd.DataFrame({"filas": {k: len(v) for k, v in datasets.items()} | {"tranco": len(tr), "ipsum": len(ips)},
                        "columnas": {k: v.shape[1] for k, v in datasets.items()} | {"tranco": tr.shape[1], "ipsum": ips.shape[1]}})
resumen

## 2. Calidad de los datos

### 2.1 Nulos por columna (%)

In [ ]:
nulos = pd.DataFrame({k: v.isna().mean().mul(100).round(1) for k, v in datasets.items()})
for k, v in datasets.items():
    n = v.isna().mean().mul(100).round(1)
    print(f"\n{k}: columnas con nulos")
    print(n[n > 0].sort_values(ascending=False).to_string() or "  (ninguna)")

Lecturas esperadas: en ThreatFox `last_seen` está vacío en la mayoría de registros, así que no sirve por sí solo para medir actividad; en TweetFeed una parte de las filas no tiene etiqueta.

### 2.2 Duplicados

In [ ]:
dup = pd.DataFrame([
    {"fuente": "threatfox", "clave": "id",                 "duplicados": tf["id"].duplicated().sum()},
    {"fuente": "threatfox", "clave": "ioc",                "duplicados": tf["ioc"].duplicated().sum()},
    {"fuente": "urlhaus",   "clave": "id",                 "duplicados": uh["id"].duplicated().sum()},
    {"fuente": "urlhaus",   "clave": "url",                "duplicados": uh["url"].duplicated().sum()},
    {"fuente": "blocklist.de", "clave": "ip",              "duplicados": bl["ip"].duplicated().sum()},
    {"fuente": "phishtank", "clave": "url (tras unescape)", "duplicados": pt["url"].duplicated().sum()},
    {"fuente": "tweetfeed", "clave": "(tipo, valor) sin derivados", "duplicados": tw[~tw["derivado"]].duplicated(["type", "value_norm"]).sum()},
])
display(dup)
display(tf[tf["ioc"].duplicated(keep=False)].sort_values("ioc")[["id", "ioc", "ioc_type", "malware_printable", "first_seen", "reporter"]].head(12))

## 3. Ventanas temporales

Cada exportación cubre un periodo distinto, lo que condiciona cualquier comparación entre fuentes.

In [ ]:
ventanas = pd.DataFrame({
    "threatfox":          [tf["first_seen"].min(), tf["first_seen"].max()],
    "urlhaus":            [uh["dateadded"].min(), uh["dateadded"].max()],
    "phishtank (todo)":   [pt["submission_time"].min(), pt["submission_time"].max()],
    f"phishtank ({VENTANA_PHISHTANK_DIAS} d)": [pt.loc[pt["reciente"], "submission_time"].min(), pt["submission_time"].max()],
    "tweetfeed":          [tw["date"].min(), tw["date"].max()],
}, index=["inicio", "fin"]).T
ventanas["dias"] = ((ventanas["fin"] - ventanas["inicio"]).dt.total_seconds() / 86400).round(1)
display(ventanas)
print("blocklist.de no incluye fechas: como fecha de observación se usa la de descarga del archivo.")

fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, (nombre, serie) in zip(axes, [("threatfox", tf["first_seen"]), ("urlhaus", uh["dateadded"]), ("tweetfeed", tw["date"])]):
    por_dia = serie.dt.floor("D").value_counts().sort_index()
    por_dia.index = por_dia.index.strftime("%m-%d")
    por_dia.plot.bar(ax=ax); ax.set_title(f"{nombre}: registros por día"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 4. ThreatFox

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tf["ioc_type"].value_counts().plot.barh(ax=axes[0], title="Tipos de IOC"); axes[0].invert_yaxis()
tf["threat_type"].value_counts().plot.barh(ax=axes[1], title="Tipo de amenaza"); axes[1].invert_yaxis()
plt.tight_layout(); plt.show()

print("confidence_level:"); display(tf["confidence_level"].value_counts().sort_index())
print("is_compromised:", tf["is_compromised"].value_counts().to_dict())
print(f"last_seen vacío: {tf['last_seen'].isna().mean():.1%}")

In [ ]:
fam = tf["malware_printable"].value_counts()
print("Familias distintas:", len(fam))
fam.head(15).plot.barh(title="Top 15 familias (malware_printable)").invert_yaxis(); plt.show()

print("Nota: 'Unknown Loader' es una categoría que actúa como 'sin clasificar':",
      fam.get("Unknown Loader", 0), "registros")

tags = tf["tags_list"].explode().dropna().value_counts()
tags.head(15).plot.barh(title="Top 15 tags (ThreatFox)").invert_yaxis(); plt.show()

In [ ]:
# Calidad del campo ioc según tipo (formato esperado)
ip_port = tf[tf["ioc_type"] == "ip:port"]["ioc"]
print("ip:port sin ':' ->", (~ip_port.str.contains(":")).sum())
print("domains con mayúsculas ->", tf[tf["ioc_type"] == "domain"]["ioc"].str.contains("[A-Z]").sum())
tf.groupby("ioc_type")["confidence_level"].describe().round(1)

## 5. URLhaus

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
uh["url_status"].value_counts().plot.bar(ax=axes[0], title="Estado de la URL")
uh["threat"].value_counts().plot.bar(ax=axes[1], title="Tipo de amenaza")
plt.tight_layout(); plt.show()

print(f"last_online vacío: {uh['last_online'].isna().mean():.1%}  |  sin tags: {(uh['tags_list'].str.len() == 0).mean():.1%}")
display(uh["tags_list"].explode().dropna().value_counts().head(15).to_frame("urls"))
display(uh["reporter"].value_counts().head(10).to_frame("urls"))

In [ ]:
# ¿El host es una IP o un dominio?
uh_es_ip = uh["host"].str.fullmatch(r"\d{1,3}(\.\d{1,3}){3}")
print(f"Hosts que son IP: {uh_es_ip.mean():.1%}   |   dominios: {(~uh_es_ip).mean():.1%}")
print("Esquemas:", uh["url"].str.extract(r"^(\w+)://")[0].value_counts().to_dict())
uh.groupby("url_status")["dateadded"].agg(["min", "max", "count"])

In [ ]:
# Concentración: ¿una sola campaña domina el dataset?
top_rep = uh["reporter"].value_counts(normalize=True).head(1)
print(f"Reportero principal: {top_rep.index[0]} = {top_rep.iloc[0]:.1%} de las URLs")
camp = uh[uh["reporter"] == top_rep.index[0]]
print("Sus tags más frecuentes:", camp["tags_list"].explode().value_counts().head(4).to_dict())
print("Sus hosts más frecuentes:", camp["host"].map(lambda h: h if not h.replace('.', '').isdigit() else '(IP)').value_counts().head(3).to_dict())

Si una única campaña concentra gran parte de las URLs, **el dataset está sesgado**: un modelo entrenado tal cual aprendería esa campaña en lugar de patrones generales. Conviene limitar el peso por reportero/campaña al construir el conjunto de entrenamiento.

## 6. blocklist.de e IPsum (IPs)

blocklist.de es la única fuente grande de IPs, pero no trae fecha ni categoría. **IPsum** no aporta indicadores propios: se usa para saber en cuántas listas independientes aparece cada IP (su nivel).

In [ ]:
FETCHED_AT_BL = pd.Timestamp(os.path.getmtime(DATA_DIR / "export-ips_all.txt"), unit="s", tz="UTC")
print(f"IPs: {len(bl):,} | únicas: {bl['ip'].nunique():,} | IPv4: {(bl['version'] == 4).sum():,} | "
      f"IPv6: {(bl['version'] == 6).sum():,} | inválidas: {(bl['version'] == 0).sum()} | privadas/reservadas: {bl['privada'].sum()}")
print("Fecha de descarga (mtime del archivo):", FETCHED_AT_BL)

# all.txt incluye /24 COMPLETOS (256 IPs seguidas) listados en bloque: no son ataques individuales
en_bloque = bl["bloque"] != ""
cuenta = bl.loc[bl["version"] == 4, "ip"].str.rsplit(".", n=1).str[0].value_counts()
print(f"Listadas en bloque (/24 con 250+ IPs): {en_bloque.sum():,} IPs en {bl.loc[en_bloque, 'bloque'].nunique()} bloques "
      f"({en_bloque.sum() / (bl['version'] == 4).sum():.1%} del IPv4)")
print("Nº de /24 según IPs listadas:", {"250+ (completos)": int((cuenta >= 250).sum()), "50-249": int(((cuenta >= 50) & (cuenta < 250)).sum()),
      "10-49": int(((cuenta >= 10) & (cuenta < 50)).sum()), "2-9": int(((cuenta >= 2) & (cuenta < 10)).sum()), "1": int((cuenta == 1).sum())})
print("Mayor /24 no completo:", cuenta[cuenta < 250].head(1).to_dict())
display(bl[en_bloque].drop_duplicates("bloque")["bloque"].str.rsplit(".", n=2).str[0].value_counts().rename_axis("/16").to_frame("bloques /24"))

tf_ips = set(tf.loc[tf["ioc_type"] == "ip:port", "ioc"].str.replace(r":\d+$", "", regex=True))
uh_ips = set(uh.loc[uh_es_ip, "host"])
tw_ips = set(tw.loc[tw["type"] == "ip", "value_norm"])
bl_ips = set(bl.loc[~en_bloque, "ip"])        # solo IPs individuales
bl_bloque_ips = set(bl.loc[en_bloque, "ip"])  # IPs listadas en bloque
print(f"Peso de blocklist.de entre los indicadores IP individuales (ThreatFox + blocklist.de + TweetFeed): {len(bl_ips) / len(tf_ips | bl_ips | tw_ips):.1%}")

In [ ]:
if len(ips):
    nivel = dict(zip(ips["ip"], ips["nivel"]))
    print(f"IPsum: {len(ips):,} IPs | IPs por nivel máximo: {ips['nivel'].value_counts().sort_index().to_dict()}")
    todos, n3 = set(ips["ip"]), set(ips.loc[ips["nivel"] >= 3, "ip"])
    filas = []
    for nombre, s in [("blocklist.de (individuales)", bl_ips), ("blocklist.de (bloques /24)", bl_bloque_ips), ("ThreatFox ip", tf_ips),
                      ("URLhaus (host IP)", uh_ips), ("TweetFeed ip", tw_ips)]:
        filas.append({"fuente": nombre, "IPs": len(s), "en IPsum ≥1": len(s & todos), "% ≥1": round(len(s & todos) / len(s) * 100, 1),
                      "en IPsum ≥3": len(s & n3), "% ≥3": round(len(s & n3) / len(s) * 100, 1)})
    display(pd.DataFrame(filas))

    # Si hay resultados de VirusTotal: ¿se comportan igual las IPs individuales y las listadas en bloque?
    vt_path = next((p for p in [DATA_DIR / "resultados_vt.csv", Path("resultados_vt.csv")] if p.exists()), None)
    if vt_path is not None:
        vt = pd.read_csv(vt_path)
        vt = vt[(vt["tipo"] == "ip") & (vt["status"] == "ok")].copy()
        vt["malicious"] = pd.to_numeric(vt["malicious"])
        vt["tipo_entrada"] = vt["indicador"].map(lambda x: "bloque" if x in bl_bloque_ips else "individual")
        display(vt.groupby("tipo_entrada").agg(n=("indicador", "size"), pct_con_1_deteccion=("malicious", lambda s: round((s >= 1).mean() * 100, 1)),
                pct_con_3_detecciones=("malicious", lambda s: round((s >= 3).mean() * 100, 1)), mediana_detecciones=("malicious", "median")))
        ind = vt[vt["tipo_entrada"] == "individual"].copy()
        ind["ipsum_nivel"] = ind["indicador"].map(nivel).fillna(0).astype(int)
        ind["grupo_ipsum"] = pd.cut(ind["ipsum_nivel"], [-1, 0, 2, 100], labels=["no está", "1-2 listas", "≥3 listas"])
        print("IPs INDIVIDUALES según el nivel de IPsum:")
        display(ind.groupby("grupo_ipsum", observed=True).agg(n=("indicador", "size"), mediana_detecciones=("malicious", "median"),
                pct_con_3_detecciones=("malicious", lambda s: round((s >= 3).mean() * 100, 1))))
        print("Correlación de Spearman nivel IPsum vs detecciones (solo individuales):", round(ind["ipsum_nivel"].corr(ind["malicious"], method="spearman"), 3))
else:
    print("Sin ficheros de IPsum en", DATA_DIR, "(pon DESCARGAR_IPSUM = True o descarga ipsum_level1..8.txt)")

Lectura: blocklist.de no tiene fecha ni categoría y mide **IPs que atacan servicios**, no infraestructura de malware. Además, **`all.txt` incluye bloques /24 completos** (256 IPs seguidas) que no son ataques individuales (probablemente redes enteras dedicadas al spam, que la propia página de exportación menciona en una nota): no tienen registro DNS propio, no aparecen en `getlast.php` y IPsum solo los recoge en 1-2 listas (alguna de las listas que agrega IPsum, quizá la propia blocklist.de, contiene esos rangos; no se puede afirmar cuál). Se tratan aparte: **IPs individuales** (observaciones reales) y **bloques** (un indicador de red por /24).

El nivel de IPsum separa muy bien las IPs **individuales** fiables de las dudosas, así que se guarda como atributo de cada IP. Los grupos de VirusTotal son pequeños y sus motores probablemente consumen listas parecidas: es una señal relativa, no una verdad.

## 7. PhishTank (URLs de phishing)

In [ ]:
ref = pt["submission_time"].max()
print("Filas:", len(pt), "| verified:", pt["verified"].value_counts().to_dict(), "| online:", pt["online"].value_counts().to_dict())
print("Ventana:", pt["submission_time"].min(), "→", ref)

edad = (ref - pt["submission_time"]).dt.days
tramos = pd.cut(edad, [-1, 7, 30, 90, 365, 10_000], labels=["≤7 d", "8–30 d", "31–90 d", "91–365 d", ">1 año"]).value_counts().sort_index()
display(tramos.to_frame("urls"))
print(f"Dentro de la ventana de {VENTANA_PHISHTANK_DIAS} días: {pt['reciente'].sum():,} URLs, {pt.loc[pt['reciente'], 'host'].nunique():,} hosts únicos")
print("URLs con '&amp;' sin limpiar:", pt["url_raw"].str.contains("&amp;").sum(), "| duplicadas tras limpiar:", pt["url"].duplicated().sum())

In [ ]:
display(pt["host"].value_counts().head(10).to_frame("urls"))
top100k = set(tr["domain"].head(100_000))
print(f"URLs cuyo host está en el top 100 000 de Tranco: {pt['host'].isin(top100k).sum():,}  (plataformas, acortadores, QR...)")
print("Hosts que son IP:", pt["host"].str.fullmatch(r"\d{1,3}(\.\d{1,3}){3}").sum())
print("Marcas suplantadas (ventana reciente):", pt.loc[pt["reciente"], "target"].value_counts().head(6).to_dict())
tf_url_set = set(tf.loc[tf["ioc_type"] == "url", "ioc"])
print("Solape de URLs con URLhaus:", pt["url"].isin(set(uh["url"])).sum(), "| con ThreatFox:", pt["url"].isin(tf_url_set).sum())

PhishTank es una **categoría nueva** (no corrobora a abuse.ch), pero: hay que quedarse con lo reciente, limpiar las entidades HTML y **no etiquetar como maliciosos los hosts de plataformas compartidas** (Google Docs, acortadores, generadores de QR...). Esas URLs solo se pueden juzgar a nivel de URL.

### 7.3 ThreatCluster (dominios y hashes de informes públicos)

## 8. TweetFeed (IOCs de investigadores en Twitter/X)

Cada fila trae reportero, etiquetas y el enlace al tweet de origen (procedencia verificable). **No está verificado** (sus términos lo dicen expresamente) y se usa con un peso graduado según la validación con VirusTotal (sección 12).

In [ ]:
unicos = tw.drop_duplicates(["type", "value_norm"])
print(f"Filas: {len(tw):,} | indicadores únicos: {len(unicos):,} | reporteros: {tw['reporter'].nunique()} | tweets: {tw['tweet'].nunique():,}")
print("Ventana:", tw["date"].min(), "→", tw["date"].max())
display(tw["type"].value_counts().to_frame("filas").join(tw[~tw["derivado"]].groupby("type").size().rename("sin_derivados"))
        .join(tw.groupby("type")["corroborado"].mean().mul(100).round(1).rename("% corroborado por sus 5 feeds")))
n_dom = (tw["type"] == "domain").sum()
print(f"Dominios que son solo el host de una URL del mismo tweet (derivados): {tw['derivado'].sum():,} de {n_dom:,} ({tw['derivado'].sum() / n_dom:.0%})")

r = tw.groupby("reporter").agg(filas=("value", "size"), pct_no_corroborado=("corroborado", lambda s: round((~s.astype(bool)).mean() * 100)))
r = r.sort_values("filas", ascending=False)
print("Concentración (top 1 / 3 / 5 / 10 reporteros):", [round(r["filas"].head(k).sum() / len(tw) * 100, 1) for k in (1, 3, 5, 10)], "%")
display(r.head(8))
print(f"Sin etiqueta: {tw['tags'].isna().mean():.1%} | etiquetas más frecuentes:", tw["tags_list"].explode().dropna().value_counts().head(8).to_dict())

In [ ]:
print("Valores no ASCII (IDN/homógrafos):", int(tw["no_ascii"].sum()), "| ejemplos de dominio en punycode:",
      tw.loc[tw["no_ascii"] & (tw["type"] == "domain"), "value_norm"].head(3).tolist())
urls = tw[tw["type"] == "url"]
print(f"URLs con IP como host: {urls['host'].map(es_ip).mean():.1%} | con host de doble uso: {urls['host'].map(es_doble_uso).mean():.1%}")
print("Hosts de doble uso más frecuentes:", urls.loc[urls["host"].map(es_doble_uso), "host"].map(registrable).value_counts().head(6).to_dict())
dv = tw.loc[tw["type"] == "domain", "value_norm"]
print("Dominios en Tranco: top 10k", int(dv.isin(set(tr["domain"].head(10_000))).sum()), "| top 100k", int(dv.isin(set(tr["domain"].head(100_000))).sum()),
      "| millón", int(dv.isin(set(tr["domain"])).sum()))

TweetFeed aporta sobre todo **cobertura nueva** (casi nada coincide con las demás fuentes) con reporteros variados. Hay que ignorar los dominios derivados de URLs, normalizar los IDN a punycode, aplicar la lista de doble uso y no tratar sus indicadores como confirmados: en la validación con VirusTotal (sección 12) su fiabilidad depende de la etiqueta y de la corroboración.

## 9. Solapamiento entre fuentes

Solo se pueden cruzar los campos con el mismo significado.

In [ ]:
tf_sha = set(tf.loc[tf["ioc_type"] == "sha256_hash", "ioc"].str.lower())
tf_md5 = set(tf.loc[tf["ioc_type"] == "md5_hash", "ioc"].str.lower())
tf_url = set(tf.loc[tf["ioc_type"] == "url", "ioc"])
tf_dom = set(tf.loc[tf["ioc_type"] == "domain", "ioc"].str.lower())
tw_x = lambda t: set(tw.loc[(tw["type"] == t) & ~tw["derivado"], "value_norm"].str.lower() if t != "url" else tw.loc[(tw["type"] == t), "value_norm"])

def fila(nombre, a, b):
    return {"cruce": nombre, "coincidencias": len(a & b), "tamaño_A": len(a), "tamaño_B": len(b)}

solape = pd.DataFrame([
    fila("TF url ∩ UH url", tf_url, set(uh["url"])),
    fila("TF dominio ∩ host UH", tf_dom, set(uh["host"])),
    fila("PhishTank url ∩ UH url", set(pt["url"]), set(uh["url"])),
    fila("PhishTank url ∩ TF url", set(pt["url"]), tf_url),
    fila("PhishTank host ∩ UH host", set(pt["host"]), set(uh["host"])),
    fila("blocklist.de ip ∩ TF ip", bl_ips, tf_ips),
    fila("TweetFeed url ∩ (UH ∪ TF ∪ PhishTank url)", tw_x("url"), set(uh["url"]) | tf_url | set(pt["url"])),
    fila("TweetFeed dominio ∩ TF dominio", tw_x("domain"), tf_dom),
    fila("TweetFeed ip ∩ (blocklist.de ∪ TF ip)", tw_x("ip"), bl_ips | tf_ips),
    fila("TweetFeed sha256 ∩ TF sha256", tw_x("sha256"), tf_sha),
    fila("TweetFeed md5 ∩ TF md5", tw_x("md5"), tf_md5),
])
solape["% de A"] = (solape["coincidencias"] / solape["tamaño_A"] * 100).round(2)
solape

El solapamiento es bajo en parte **por construcción** (ventanas distintas) y porque las fuentes cubren amenazas **distintas** (malware, phishing, ataques a servicios, informes de la comunidad): se complementan más que se corroboran.

## 10. Tranco y la clase no maliciosa

Tranco mide **popularidad**, no **benignidad**. Antes de usarlo como etiqueta negativa hay que comprobar cuánto choca con las fuentes maliciosas.

In [ ]:
print(tr.head(3).to_string(index=False))
print("rank monótono:", tr["rank"].is_monotonic_increasing, "| duplicados:", tr["domain"].duplicated().sum(), "| nulos:", tr.isna().sum().sum())

try:   # dominio registrable con la Public Suffix List incluida en el paquete (sin red)
    import tldextract
    _ext = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None)
    def registrable(host):
        r = _ext(host); return r.top_domain_under_public_suffix if hasattr(r, "top_domain_under_public_suffix") else r.registered_domain
    print("Usando tldextract")
except Exception:
    def registrable(host):
        p = host.split("."); return ".".join(p[-2:]) if len(p) >= 2 else host
    print("tldextract no disponible: se aproxima con las dos últimas etiquetas (impreciso en .co.uk, etc.)")

In [ ]:
benignos = set(tr["domain"])
uh_dom = uh.loc[~uh_es_ip, "host"]
tf_url_hosts = pd.Series(list(tf_url)).map(lambda u: pd.Series([u]).str.extract(r"^\w+://([^/:?#]+)")[0].iloc[0].lower())
pt_h = pt.loc[pt["reciente"] & ~pt["host"].map(es_ip), "host"].drop_duplicates()
tw_h = pd.concat([tw.loc[tw["type"] == "url", "host"], tw.loc[(tw["type"] == "domain") & ~tw["derivado"], "value_norm"]]).drop_duplicates()
tw_h = tw_h[~tw_h.map(es_ip) & (tw_h != "")]

cruce = pd.DataFrame([{"origen": nombre, "total": len(s), "en_tranco_exacto": s.isin(benignos).sum(),
                       "en_tranco_registrable": s.map(registrable).isin(benignos).sum()}
                      for nombre, s in [("URLhaus (host, solo dominios)", uh_dom), ("ThreatFox domain", pd.Series(sorted(tf_dom))),
                                        ("ThreatFox url (host)", tf_url_hosts), (f"PhishTank hosts ({VENTANA_PHISHTANK_DIAS} d)", pt_h),
                                        ("TweetFeed hosts", tw_h)]])
display(cruce)
print("Dominios de Tranco más repetidos entre las URLs maliciosas de URLhaus:")
display(uh_dom[uh_dom.isin(benignos)].value_counts().head(10).to_frame("urls_maliciosas"))

Dominios como `github.com` aparecen en Tranco **y** alojan malware: son de **doble uso**. Propuesta para construir la clase no maliciosa:

1. Tomar solo el **top N** (p. ej. 50 000), no el millón completo.
2. **Excluir** todo dominio que aparezca en las fuentes maliciosas.
3. **Excluir** servicios de alojamiento/compartición y **plataformas de subdominios** (`DOBLE_USO` en `cargadores_corregidos.py`): cualquier host bajo ellas es de un usuario y se dejan fuera de ambas clases a nivel de dominio (siguen siendo válidas como URLs o hosts).
4. Trabajar **a nivel de dominio** (extrayendo el host de las URLs), para no enseñar al modelo a distinguir «URL con ruta» de «dominio».

In [ ]:
TOP_N = 50_000
maliciosos_dom = (set(uh_dom.map(registrable)) | set(pd.Series(sorted(tf_dom)).map(registrable)) | set(tf_url_hosts.map(registrable))
                  | set(pt_h.map(registrable)) | set(tw_h.map(registrable)))
# Los hosts que son IP (p. ej. URLs de ThreatFox) tienen registrable vacío: sin esto, todo dominio de Tranco que es
# un sufijo público (bet.br) contaría como choque. Criterio común con ensayo_aprendizaje.py.
maliciosos_dom.discard("")
candidatos = tr.head(TOP_N).copy()
candidatos["registrable"] = candidatos["domain"].map(registrable)
candidatos["choca_con_maliciosos"] = candidatos["registrable"].isin(maliciosos_dom)
candidatos["doble_uso"] = candidatos["domain"].map(es_doble_uso)
limpios = candidatos[~candidatos["choca_con_maliciosos"] & ~candidatos["doble_uso"]]
print(f"Top {TOP_N:,}: descartados por chocar: {candidatos['choca_con_maliciosos'].sum():,}, doble uso: {candidatos['doble_uso'].sum():,} "
      f"-> clase no maliciosa: {len(limpios):,}")

# Dominios maliciosos utilizables por el modelo A (nivel dominio): sin IPs ni plataformas de doble uso
cand_mal = pd.concat([uh_dom, pd.Series(sorted(tf_dom)), tf_url_hosts, pt_h, tw_h]).str.lower().drop_duplicates()
cand_mal = cand_mal[~cand_mal.map(es_ip) & ~cand_mal.map(es_doble_uso)]
print(f"Dominios maliciosos únicos utilizables (sin doble uso): {len(cand_mal):,}  |  no maliciosos: {len(limpios):,}")
print("Dominios registrables únicos entre los maliciosos:", cand_mal.map(registrable).nunique())

## 11. Esquema común (prototipo)

Normalización mínima para el modelo `indicator` / `sighting` de la base de datos: tipos unificados, valores en minúsculas, `ip:port` separado, una fila por **observación** (fuente + registro original) y tres campos de contexto:

- **`category`**: qué mide cada fuente (`malware_distribution`, `c2`, `phishing`, `attack_source`, `attack_network`, `malware`...). La severidad solo debe compararse, o ponderarse, dentro de una misma categoría.
- **`source_family`**: abuse.ch (ThreatFox + URLhaus) cuenta como una. La corroboración se cuenta por **familias**, no por observaciones.
- **`ipsum_nivel`**: nº de listas de IPsum en las que aparece una IP (señal de fiabilidad; 0 si no aparece).

In [ ]:
MAPA_TIPO = {"sha256_hash": "sha256", "md5_hash": "md5", "sha1_hash": "sha1", "domain": "domain", "url": "url", "ip:port": "ip"}
MAPA_CATEGORIA_TF = {"botnet_cc": "c2", "payload_delivery": "malware_distribution", "payload": "malware_distribution", "cc_skimming": "skimming"}
COLS = ["source", "source_family", "source_record_id", "type", "value_norm", "seen_at", "category", "family",
        "confidence", "status", "reporter", "corroborado"]
nivel_ipsum = dict(zip(ips["ip"], ips["nivel"]))

t = tf.copy()
t["type"] = t["ioc_type"].map(MAPA_TIPO)
t["value_norm"] = t["ioc"].str.strip().where(t["type"].isin(["url"]), t["ioc"].str.strip().str.lower())
t.loc[t["type"] == "ip", "value_norm"] = t.loc[t["type"] == "ip", "ioc"].str.replace(r":\d+$", "", regex=True)
s_tf = pd.DataFrame({"source": "threatfox", "source_family": "abuse.ch", "source_record_id": t["id"].astype(str), "type": t["type"],
                     "value_norm": t["value_norm"], "seen_at": t["first_seen"], "category": t["threat_type"].map(MAPA_CATEGORIA_TF).fillna("other"),
                     "family": t["malware_printable"], "confidence": t["confidence_level"], "status": None, "reporter": t["reporter"]})
s_uh = pd.DataFrame({"source": "urlhaus", "source_family": "abuse.ch", "source_record_id": uh["id"].astype(str), "type": "url",
                     "value_norm": uh["url"].str.strip(), "seen_at": uh["dateadded"], "category": "malware_distribution",
                     "family": uh["tags_list"].map(lambda l: l[0] if l else None), "confidence": None, "status": uh["url_status"], "reporter": uh["reporter"]})
bli = bl[bl["bloque"] == ""]
s_bl = pd.DataFrame({"source": "blocklist.de", "source_family": "blocklist.de", "source_record_id": bli["ip"], "type": "ip",
                     "value_norm": bli["ip"], "seen_at": FETCHED_AT_BL, "category": "attack_source",
                     "family": None, "confidence": None, "status": None, "reporter": None})
blk = bl[bl["bloque"] != ""].drop_duplicates("bloque")  # los /24 completos son un indicador de red, no 256 IPs
s_blk = pd.DataFrame({"source": "blocklist.de", "source_family": "blocklist.de", "source_record_id": blk["bloque"], "type": "cidr",
                      "value_norm": blk["bloque"], "seen_at": FETCHED_AT_BL, "category": "attack_network",
                      "family": None, "confidence": None, "status": None, "reporter": None})
p = pt[pt["reciente"]].drop_duplicates("url")
s_pt = pd.DataFrame({"source": "phishtank", "source_family": "phishtank", "source_record_id": p["phish_id"].astype(str), "type": "url",
                     "value_norm": p["url"].str.strip(), "seen_at": p["submission_time"], "category": "phishing",
                     "family": p["target"].where(p["target"] != "Other"), "confidence": None, "status": "online", "reporter": None})

def categoria_tweetfeed(tags):
    t_ = {x.lower() for x in tags}
    if t_ & {"#phishing", "#scam"}: return "phishing"
    if "#c2" in t_: return "c2"
    return "malware" if t_ else "unspecified"

w = tw[~tw["derivado"]]
s_tw = pd.DataFrame({"source": "tweetfeed", "source_family": "tweetfeed", "source_record_id": w["tweet"] + "|" + w["value"],
                     "type": w["type"], "value_norm": w["value_norm"].where(w["type"] == "url", w["value_norm"].str.lower()),
                     "seen_at": w["date"], "category": w["tags_list"].map(categoria_tweetfeed), "family": None,
                     "confidence": None, "status": None, "reporter": w["reporter"]})
s_tw["corroborado"] = w["corroborado"].astype(bool).values

sightings = pd.concat([s_tf, s_uh, s_bl, s_blk, s_pt, s_tw], ignore_index=True)
sightings["corroborado"] = sightings["corroborado"].astype("boolean")
sightings = sightings[COLS]
sightings["ipsum_nivel"] = sightings["value_norm"].map(nivel_ipsum).where(sightings["type"] == "ip").fillna(0).where(sightings["type"] == "ip")

indicadores = sightings.groupby(["type", "value_norm"]).agg(
    n_sightings=("source", "size"), n_fuentes=("source", "nunique"), n_familias=("source_family", "nunique"),
    ipsum_nivel=("ipsum_nivel", "max"), categorias=("category", lambda s: sorted(set(s))),
    first_seen=("seen_at", "min"), last_seen=("seen_at", "max")).reset_index()

print(f"Observaciones (sightings): {len(sightings):,}  |  indicadores únicos: {len(indicadores):,}")
display(sightings["type"].value_counts().to_frame("sightings").join(indicadores["type"].value_counts().rename("indicadores")))
display(sightings.groupby(["source_family", "category"]).size().to_frame("sightings"))
display(indicadores["n_familias"].value_counts().sort_index().rename("indicadores").to_frame())
print("Peso de cada familia de fuentes entre los indicadores de tipo IP:")
print(sightings.loc[sightings["type"] == "ip"].groupby("source_family")["value_norm"].nunique().pipe(lambda s: (s / s.sum()).round(3)).to_dict())
if len(ips):
    ip_ind = indicadores[indicadores["type"] == "ip"]
    print("IPs por nivel de IPsum:", pd.cut(ip_ind["ipsum_nivel"], [-1, 0, 2, 100], labels=["no está", "1-2 listas", "≥3 listas"]).value_counts().sort_index().to_dict())

## 12. Validación con VirusTotal

Una muestra reproducible (`vt_muestreo.py`, semilla 42) de IPs, dominios y, si se han consultado, URLs completas y hashes (grupos `url_*` y `hash_*`) se consulta a la API de VirusTotal. En URLs y hashes, `not_found` significa que VirusTotal nunca los ha analizado. Un indicador se considera **respaldado** con ≥1 detección y **confirmado** con ≥3 (con una sola, el ruido de un motor aislado es alto). Los intervalos son de Wilson al 95 %. Las IPs se separan en individuales y listadas en bloque /24. VirusTotal no es una verdad absoluta: sus motores consumen feeds como estos y cero detecciones no equivale a benigno.

In [ ]:
import numpy as np

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round((c - h) * 100), round((c + h) * 100)

vt_path = next((p for p in [DATA_DIR / "resultados_vt.csv", Path("resultados_vt.csv"), Path("../src/resultados_vt.csv")]
                if p.exists()), None)
if vt_path is None:
    print("No hay resultados_vt.csv: ejecuta vt_muestreo.py (ver el documento de fase).")
else:
    vt = pd.read_csv(vt_path)
    print("Estados:", vt["status"].value_counts().to_dict())
    nf_grupo = vt.loc[vt["status"] == "not_found", "grupo"].value_counts()  # VirusTotal nunca lo analizó
    vt = vt[vt["status"] == "ok"].copy()
    vt["malicious"] = pd.to_numeric(vt["malicious"])
    plataforma = (vt["tipo"] == "domain") & vt["indicador"].map(es_doble_uso)
    print(f"Consultas válidas: {len(vt)} | excluidas por doble uso: {int(plataforma.sum())}")
    vt = vt[~plataforma].copy()
    vt.loc[vt["tipo"] == "ip", "grupo"] = vt.loc[vt["tipo"] == "ip", "indicador"].map(lambda x: "ip_bloque" if x in bl_bloque_ips else "ip_individual")
    vt.loc[vt["grupo"].str.startswith("dom_tranco"), "grupo_tabla"] = "dom_tranco (total)"
    vt["grupo_tabla"] = vt["grupo_tabla"].fillna(vt["grupo"])
    filas = []
    for grupo, g in vt.groupby("grupo_tabla"):
        n, k1, k3 = len(g), int((g["malicious"] >= 1).sum()), int((g["malicious"] >= 3).sum())
        nf_ = int(nf_grupo.get(grupo, 0))
        filas.append({"grupo": grupo, "n": n, "not_found": nf_, "cobertura": f"{n / (n + nf_):.0%}", "≥1 detección": f"{k1 / n:.0%}", "IC ≥1": "%d–%d" % wilson(k1, n),
                      "≥3 detecciones": f"{k3 / n:.0%}", "IC ≥3": "%d–%d" % wilson(k3, n), "mediana": g["malicious"].median()})
    display(pd.DataFrame(filas))
    tr_ = vt[vt["grupo"].str.startswith("dom_tranco")]
    print("Dominios de Tranco con alguna detección:", tr_.loc[tr_["malicious"] >= 1].sort_values("malicious", ascending=False)["indicador"].head(12).tolist())
    for g_ in ("dom_abusech", "dom_phishtank", "dom_tweetfeed"):
        print(f"{g_}: sin ninguna detección ->", vt.loc[(vt["grupo"] == g_) & (vt["malicious"] == 0), "indicador"].tolist())

### 12.1 TweetFeed: la fiabilidad depende de la etiqueta y de la corroboración

Cada dominio de la muestra se cruza con `month.csv` para ver su etiqueta, su reportero y si sus 5 feeds de referencia lo corroboran.

In [ ]:
if vt_path is not None and "dom_tweetfeed" in set(vt["grupo"]):
    filas = []
    for _, x in tw.iterrows():
        h_ = x["host"] if x["type"] == "url" else (x["value_norm"] if x["type"] == "domain" else None)
        if h_:
            filas.append((h_, x["reporter"], categoria_tweetfeed(x["tags_list"]), bool(x["corroborado"])))
    m_ = pd.DataFrame(filas, columns=["host", "reporter", "categoria", "corroborado"])
    info = m_.groupby("host").agg(reporter=("reporter", lambda s: s.value_counts().index[0]), categoria=("categoria", lambda s: s.value_counts().index[0]),
                                  corroborado=("corroborado", "max")).reset_index()
    t_ = vt[vt["grupo"] == "dom_tweetfeed"].merge(info, left_on="indicador", right_on="host", how="left")
    print(f"Dominios de TweetFeed en la muestra: {len(t_)} | con correspondencia en month.csv: {int(t_['host'].notna().sum())}")
    for col in ("categoria", "corroborado"):
        res = t_.groupby(col).agg(n=("indicador", "size"), k1=("malicious", lambda s: int((s >= 1).sum())), k3=("malicious", lambda s: int((s >= 3).sum())))
        res["% ≥3"] = (res["k3"] / res["n"] * 100).round(0)
        res["IC ≥3"] = [("%d–%d" % wilson(a, b)) for a, b in zip(res["k3"], res["n"])]
        display(res)
    display(t_.groupby("reporter").agg(n=("indicador", "size"), k3=("malicious", lambda s: int((s >= 3).sum()))).sort_values("n", ascending=False).head(6))
else:
    print("Sin resultados de VirusTotal para TweetFeed.")

Lectura: con etiqueta de **malware o C2** (94 %, n=18), o **corroborados** por sus feeds de referencia (100 %, n=11), los dominios de TweetFeed se confirman casi siempre; con etiqueta de **phishing** (60 %, n=30) y, sobre todo, **sin etiqueta** (28 %, n=18), mucho menos. Por eso se le da un peso graduado, no uno único. Los grupos por etiqueta son pequeños (intervalos amplios): es una pauta, no una ley.

### 12.2 URLs completas y hashes

Las URLs pueden estar en plataformas de doble uso: se juzgan como URL, no como dominio. Se comparan según el tipo de host
(dominio propio, plataforma de doble uso o IP) y, en TweetFeed, URLs y hashes según etiqueta y corroboración, como en los
dominios. `cobertura` (tabla de la sección 12) es la parte de cada grupo que VirusTotal conoce.

In [ ]:
def tasas(d, col):
    res = d.groupby(col).agg(n=("indicador", "size"), k1=("malicious", lambda s: int((s >= 1).sum())),
                             k3=("malicious", lambda s: int((s >= 3).sum())))
    res["% ≥3"] = (res["k3"] / res["n"] * 100).round(0)
    res["IC ≥3"] = [("%d–%d" % wilson(a, b)) for a, b in zip(res["k3"], res["n"])]
    return res

if vt_path is not None and vt["grupo"].str.startswith("url_").any():
    u_ = vt[(vt["tipo"] == "url") & (vt["clase_esperada"] == "malicioso")].copy()
    host_ = u_["indicador"].str.extract(r"^https?://([^/:?#]+)", expand=False).str.lower()
    u_["host"] = "dominio propio"
    u_.loc[host_.map(es_doble_uso), "host"] = "plataforma doble uso"
    u_.loc[host_.map(es_ip), "host"] = "IP"
    display(tasas(u_, ["grupo", "host"]))
    print("URLs maliciosas sin confirmar (<3 detecciones):")
    display(u_.loc[u_["malicious"] < 3, ["grupo", "malicious", "indicador"]])
    print("Raíces de Tranco con alguna detección:",
          vt.loc[(vt["grupo"] == "url_tranco_raiz") & (vt["malicious"] >= 1), "indicador"].tolist())

    tw["categoria"] = tw["tags_list"].map(categoria_tweetfeed)
    for tipo_, grupo_, clave_ in (("url", "url_tweetfeed", "value"), ("sha256", "hash_tweetfeed", "value_norm")):
        info_ = tw[tw["type"] == tipo_].groupby(clave_).agg(categoria=("categoria", "first"), corroborado=("corroborado", "max"))
        d_ = vt[vt["grupo"] == grupo_].merge(info_, left_on="indicador", right_index=True, how="left")
        print(f"{grupo_}: {len(d_)} con análisis")
        display(tasas(d_, "categoria"))
        display(tasas(d_, "corroborado"))
    h_ = vt[vt["grupo"].str.startswith("hash_")]
    print("Hashes sin confirmar:")
    display(h_.loc[h_["malicious"] < 3, ["grupo", "malicious", "undetected", "categorias", "indicador"]])
else:
    print("Sin resultados de VirusTotal para URLs y hashes.")

Lectura: una URL concreta se confirma más que el dominio de la misma fuente (URLhaus 73 % frente a 59 % de abuse.ch;
TweetFeed 82 % frente a 61 %), porque su host puede ser un sitio legítimo comprometido o una plataforma; aun así, en
plataformas de doble uso la URL sigue siendo juzgable (PhishTank 14 de 14). Las URLs sin confirmar son sobre todo
ficheros en GitHub o Google Drive, binarios de botnet servidos desde IPs e instaladores legítimos (ScreenConnect) usados
como señuelo: lo malicioso es el contenido, no el host. En hashes, ThreatFox se confirma al 100 % (en parte circular:
abuse.ch y VirusTotal comparten muestras) y TweetFeed al 91 %, con un 10 % de hashes que VirusTotal no conoce; los
corroborados se confirman siempre (n=23). Los grupos son de 30–50 indicadores: intervalos de ±10–15 puntos.

## 13. Conclusiones para el diseño

- **Conjunto final:** ThreatFox, URLhaus, blocklist.de, PhishTank y TweetFeed como fuentes; IPsum como señal de fiabilidad de IPs; Tranco para la clase no maliciosa. El resto se descartó por aportar poco o añadir complejidad (anexo).
- **Validación con VirusTotal (sección 12, 825 indicadores):** PhishTank es la más fiable; TweetFeed y abuse.ch quedan parecidas en dominios (≈60 % con ≥3 detecciones), pero TweetFeed se puede graduar por corroboración y etiqueta; en IPs, el nivel de IPsum separa muy bien las individuales y los bloques /24 son más débiles. Las URLs se confirman más que sus dominios (73–100 %) y los hashes casi siempre (91–100 %), aunque VirusTotal no conoce el 10 % de los de TweetFeed.
- **Cargadores:** abuse.ch, PhishTank y TweetFeed requieren tratamiento propio (cabecera `# id`, `n/a`/`None`, fechas con `UTC`, tags como lista, `&amp;` (incluido el doble escape, sin `html.unescape`), dominios derivados, IDN a punycode).
- **Duplicados:** ThreatFox repite algunos IOCs con `id` distinto → la unicidad se garantiza sobre `(tipo, valor normalizado)`.
- **Ventanas distintas:** PhishTank trae histórico desde 2011 (se filtra a una ventana reciente) y blocklist.de no trae fechas. La ausencia de un IOC en una fuente no implica inactividad.
- **Solapamiento bajo:** las fuentes cubren amenazas distintas; la corroboración por familias es escasa. La única corroboración útil para IPs viene de IPsum.
- **Bloques en blocklist.de:** `all.txt` incluye /24 completos (≈ un tercio del IPv4) que no son ataques individuales; se separan y se representan como indicadores de red (`cidr`, categoría `attack_network`), y las estadísticas de fiabilidad se calculan solo con IPs individuales.
- **Sesgos de concentración:** una campaña domina URLhaus, blocklist.de domina las IPs individuales y las plataformas compartidas dominan PhishTank y TweetFeed → límites de peso por fuente y campaña, y lista de doble uso.
- **Categoría de amenaza:** hace falta `category`; el modelo A solo puntúa dominios y los hashes/IPs dependen de reglas.
- **Clase no maliciosa:** top N de Tranco depurado (sin choques ni doble uso) y entrenamiento a nivel de dominio.
- **Siguientes pasos:** crear el DDL de PostgreSQL (con el tipo `cidr`), automatizar la ingesta (con las claves fuera del repositorio), entrenar el modelo A y fijar los pesos de las reglas, decidiendo si URLs y hashes llevan pesos propios.

## Anexo. Fuentes evaluadas y descartadas

| Fuente | Motivo |
|---|---|
| MalwareBazaar | Solo hashes (1 383 muestras en 2 días), sin uso en el modelo y sin validar; el formato añade complejidad (cargador en `cargadores_opcionales.py`) |
| ThreatCluster | ≈1,7 % de los indicadores, solape casi nulo, confianza constante y falsos positivos de extracción (cargador en `cargadores_opcionales.py`) |
| OTX | 754 registros, fechas antiguas y esquema distinto |
| Feodo Tracker | Conjuntos vacíos tras los derribos policiales |
| Spamhaus (listas DNS) | Acceso gratuito de 30 días: solo útil para validar una muestra |
| `IoCs_Identifiers.json` | 200 dominios alfabéticos, subconjunto del top 10 000 de Tranco y sin veredictos |
| Filtro de Bloom de CIRCL | 0,7–1 GB; solo indica «conocido», no «benigno»; sin uso para entrenar |
| MISP como vía principal | Duplicaría datos de abuse.ch y añade servicios |
| vciso.au | Agregador de otros feeds; datos no accesibles; licencia desconocida |
| C2IntelFeeds / C2-Tracker | Volumen pequeño, etiquetas «Possible», sin fechas; una URL del catálogo ya no existe |
| FireHOL level1 | Lista de rangos, no de indicadores |
| Otras listas de IPs de ataque (GreenSnow, CINS, Emerging Threats...) | Misma semántica que blocklist.de; IPsum ya las resume |